# Building Stateful AI Workflow

In [162]:
from langgraph.graph import StateGraph

#### 1. States

In [163]:
from typing import TypedDict,Optional

class AuthState(TypedDict):
    username:Optional[str]
    password:Optional[str]
    is_authenticated:Optional[bool]
    output:Optional[str]
    
    

#### Example Objects and Their States

#### object 1: Successfull Login

In [164]:
auth_state_1: AuthState={
    'username':'ram_yadav',
    'password':'ram@123',
    'is_authenticated':True,
    'output':'login successful'
}

print(f'auth_state_1:{auth_state_1}')

auth_state_1:{'username': 'ram_yadav', 'password': 'ram@123', 'is_authenticated': True, 'output': 'login successful'}


#### object 2: Unsuccessful Login

In [165]:
auth_state_2: AuthState={
    'username':'',
    'password':'shyam@123',
    'is_authenticated':False,
    'output':'Authentication failed. Please try again'
}
print(f'auth_state_2: {auth_state_2}')

auth_state_2: {'username': '', 'password': 'shyam@123', 'is_authenticated': False, 'output': 'Authentication failed. Please try again'}


#### 2. Nodes

In [166]:
def input_node(state):
    print(state)
    
    if state.get('username','')=='':
        username=input('What is your username?')
    
    password=input('Enter your password')
    
    if state.get('username','')=='':
        return {'username':username,'password':password}
    else:
        return {'password':password}

In [167]:
input_node(auth_state_1)

{'username': 'ram_yadav', 'password': 'ram@123', 'is_authenticated': True, 'output': 'login successful'}


{'password': 'dsds'}

In [168]:
input_node(auth_state_2)

{'username': '', 'password': 'shyam@123', 'is_authenticated': False, 'output': 'Authentication failed. Please try again'}


{'username': 'sdfasd', 'password': 'sdf'}

#### Validate Credentials Node

In [169]:
def validate_credentials_node(state):
    username=state.get('username','')
    password=state.get('password')
    
    print(f'username:{username} , password:{password}')
    # simulated credentials validations
    if username=='test_user' and password=='secure_password':
        is_authenticated=True
    else:
        is_authenticated=False
    return{'is_authenticated':is_authenticated}

#### Incorrect Format

In [170]:
validate_credentials_node(auth_state_1)

username:ram_yadav , password:ram@123


{'is_authenticated': False}

#### Correct Format

In [171]:
auth_state_3:AuthState={
    'username':'test_user',
    'password':'secure_password',
    'is_authenticated':True,
    'output':'Authentications Failed. Please try again'
}
print(f'auth_state_3:{auth_state_3}')

auth_state_3:{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Authentications Failed. Please try again'}


In [172]:
validate_credentials_node(auth_state_3)

username:test_user , password:secure_password


{'is_authenticated': True}

#### Defining the Success Node

In [173]:
def success_node(state):
    return {'output':'Autentications successful! Welcome'}

In [174]:
success_node(auth_state_3)

{'output': 'Autentications successful! Welcome'}

#### Define failure node

In [175]:
def failure_node(state):
    return {'output':'Not Successful, please try again'}

#### Defining the router node

In [176]:
def router(state):
    if state['is_authenticated']:
        return 'success_node'
    else:
        return 'failure_node'

#### Creating the graph

In [177]:
from langgraph.graph import StateGraph
from langgraph.graph import END

In [178]:
# creating an instances of StateGraph with the GraphState structure
workflow=StateGraph(AuthState)
workflow

#### Adding nodes to the graph

In [179]:
workflow.add_node('InputNode',input_node)
workflow.add_node('ValidateCredential',validate_credentials_node)
workflow.add_node('Success',success_node)
workflow.add_node('Failure',failure_node)


#### Edges

In [180]:
workflow.add_edge("InputNode","ValidateCredential")
workflow.add_edge('Success',END)
workflow.add_edge('Failure','InputNode')

#### Building an Authentication Workflow

In [181]:
workflow.add_conditional_edges('ValidateCredential',router,{'success_node':'Success','Failure_node':'Failure'})

#### Setting the Entry Point

In [182]:
workflow.set_entry_point('InputNode')

#### Compiling the Workflow

In [183]:
app=workflow.compile()

In [184]:
inputs={"username":'test_user'}
result=app.invoke(inputs)
print(result)

{'username': 'test_user'}
username:test_user , password:secure_password
{'username': 'test_user', 'password': 'secure_password', 'is_authenticated': True, 'output': 'Autentications successful! Welcome'}
